# ДЗ №14 — Transformer embeddings: BERT / DistilBERT + external classifiers

Датасеты:
1. `emotion`
2. `20_newsgroups(4)` 

Задание:
1. Берём две разные предобученные transformer-модели:
   - `bert-base-uncased`
   - `distilbert-base-uncased`
2. Получаем CLS-эмбеддинги текстов
3. Обучаем внешние классификаторы:
   - Logistic Regression
   - Linear SVM
4. Пробуем готовые fine-tuned модели с HuggingFace:
   - для `emotion`: `bhadresh-savani/bert-base-uncased-emotion`
   - для `20_newsgroups`: `rjac/bert-20news-classification`
5. Для fine-tuned моделей проверяем два варианта:
   - CLS из fine-tuned модели → внешний классификатор
   - встроенный классификатор fine-tuned модели
6. Сравниваем все подходы по `F1 macro`


## 1. Импорты и настройки

In [14]:
!pip install transformers datasets scikit-learn pandas numpy torch tqdm

import os
import random
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
)

from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [15]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 2. Общие параметры эксперимента

In [16]:
# None, чтобы использовать все тексты
TRAIN_EMBED_LIMIT = 3000
TEST_EMBED_LIMIT = 1000

BATCH_SIZE = 16

BASE_TRANSFORMER_MODELS = [
    "bert-base-uncased",
    "distilbert-base-uncased",
]

FINE_TUNED_MODELS = {
    "emotion": "bhadresh-savani/bert-base-uncased-emotion",
    "20_newsgroups(4)": "rjac/bert-20news-classification",
}

external_classifiers = {
    "LogisticRegression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED)
    ),
    "LinearSVM": make_pipeline(
        StandardScaler(),
        LinearSVC(random_state=SEED)
    ),
}

## 3. Вспомогательные функции

In [17]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def limit_data(texts, labels, limit=None):
    texts = list(texts)
    labels = list(labels)
    if limit is None:
        return texts, np.array(labels)
    return texts[:limit], np.array(labels[:limit])


def get_cls_embeddings(texts, model_name, max_length=128, batch_size=16, from_tf=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, from_tf=from_tf).to(device)
    model.eval()

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)

        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)


def train_external_classifiers(X_train, y_train, X_test, y_test, dataset_name, representation_name):
    rows = []
    for clf_name, clf in external_classifiers.items():
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)

        rows.append({
            "dataset": dataset_name,
            "representation": representation_name,
            "classifier": clf_name,
            "f1_macro": f1_score(y_test, pred, average="macro"),
            "f1_micro": f1_score(y_test, pred, average="micro"),
            "f1_weighted": f1_score(y_test, pred, average="weighted"),
        })
    return rows


def predict_with_finetuned_model(texts, model_name, label_names, max_length=128, batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    id2label = model.config.id2label
    print("Model id2label:", id2label)

    label_names_lower = [x.lower() for x in label_names]
    preds_local = []

    for start in tqdm(range(0, len(texts), batch_size), desc=f"Direct classifier: {model_name}"):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            logits = model(**encoded).logits
            pred_ids = torch.argmax(logits, dim=1).cpu().numpy()

        for pred_id in pred_ids:
            raw_label = str(id2label.get(int(pred_id), f"LABEL_{pred_id}")).lower()

            # Случай 1: label задан текстом
            if raw_label in label_names_lower:
                preds_local.append(label_names_lower.index(raw_label))
            # Случай 2: label задан как LABEL_0, LABEL_1
            elif raw_label.startswith("label_"):
                idx = int(raw_label.split("_")[-1])
                if idx < len(label_names):
                    preds_local.append(idx)
                else:
                    preds_local.append(-1)
            else:
                preds_local.append(-1)

    return np.array(preds_local)

## 4. Загрузка датасетов

In [18]:
# Emotion dataset
emotion_dataset = load_dataset("emotion")

emotion_label_names = emotion_dataset["train"].features["label"].names

emotion_train_texts_full = [clean_text(t) for t in emotion_dataset["train"]["text"]]
emotion_train_labels_full = list(emotion_dataset["train"]["label"])

emotion_test_texts_full = [clean_text(t) for t in emotion_dataset["test"]["text"]]
emotion_test_labels_full = list(emotion_dataset["test"]["label"])

emotion_train_texts, emotion_train_labels = limit_data(
    emotion_train_texts_full,
    emotion_train_labels_full,
    TRAIN_EMBED_LIMIT
)
emotion_test_texts, emotion_test_labels = limit_data(
    emotion_test_texts_full,
    emotion_test_labels_full,
    TEST_EMBED_LIMIT
)

print("Emotion labels:", emotion_label_names)
print("Emotion train/test:", len(emotion_train_texts), len(emotion_test_texts))

Emotion labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Emotion train/test: 3000 1000


In [19]:
# 20_newsgroups(4)
news_dataset = load_dataset("SetFit/20_newsgroups")

news_categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x",
]

train_news = news_dataset["train"].filter(lambda x: x["label_text"] in news_categories)
test_news = news_dataset["test"].filter(lambda x: x["label_text"] in news_categories)

news_label2id = {label: i for i, label in enumerate(news_categories)}
news_id2label = {i: label for label, i in news_label2id.items()}

news_train_texts_full = [clean_text(t) for t in train_news["text"]]
news_train_labels_full = [news_label2id[label] for label in train_news["label_text"]]

news_test_texts_full = [clean_text(t) for t in test_news["text"]]
news_test_labels_full = [news_label2id[label] for label in test_news["label_text"]]

news_train_texts, news_train_labels = limit_data(
    news_train_texts_full,
    news_train_labels_full,
    TRAIN_EMBED_LIMIT
)
news_test_texts, news_test_labels = limit_data(
    news_test_texts_full,
    news_test_labels_full,
    TEST_EMBED_LIMIT
)

print("20_newsgroups labels:", news_categories)
print("20_newsgroups train/test:", len(news_train_texts), len(news_test_texts))

Repo card metadata block was not found. Setting CardData to empty.


20_newsgroups labels: ['comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.graphics', 'comp.windows.x']
20_newsgroups train/test: 2345 1000


# Часть 1. CLS из обычных предобученных моделей → внешний классификатор

В этой части используем обычные предобученные модели, которые не дообучались на наших датасетах:

- `bert-base-uncased`
- `distilbert-base-uncased`

Из каждой модели достаём CLS-вектор текста и обучаем два внешних классификатора:

- Logistic Regression
- Linear SVM

In [21]:
all_results = []

for model_name in BASE_TRANSFORMER_MODELS:
    print("Base model:", model_name)

    # Emotion
    X_train_emb_emotion = get_cls_embeddings(
        emotion_train_texts,
        model_name,
        max_length=128,
        batch_size=BATCH_SIZE
    )
    X_test_emb_emotion = get_cls_embeddings(
        emotion_test_texts,
        model_name,
        max_length=128,
        batch_size=BATCH_SIZE
    )

    all_results.extend(
        train_external_classifiers(
            X_train_emb_emotion,
            emotion_train_labels,
            X_test_emb_emotion,
            emotion_test_labels,
            dataset_name="emotion",
            representation_name=f"CLS_{model_name}"
        )
    )

    # 20_newsgroups
    X_train_emb_news = get_cls_embeddings(
        news_train_texts,
        model_name,
        max_length=256,
        batch_size=BATCH_SIZE
    )
    X_test_emb_news = get_cls_embeddings(
        news_test_texts,
        model_name,
        max_length=256,
        batch_size=BATCH_SIZE
    )

    all_results.extend(
        train_external_classifiers(
            X_train_emb_news,
            news_train_labels,
            X_test_emb_news,
            news_test_labels,
            dataset_name="20_newsgroups(4)",
            representation_name=f"CLS_{model_name}"
        )
    )

base_results_df = pd.DataFrame(all_results).sort_values(
    ["dataset", "f1_macro"],
    ascending=[True, False]
)

base_results_df

Base model: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,dataset,representation,classifier,f1_macro,f1_micro,f1_weighted
6,20_newsgroups(4),CLS_distilbert-base-uncased,LogisticRegression,0.643699,0.643,0.643449
2,20_newsgroups(4),CLS_bert-base-uncased,LogisticRegression,0.620788,0.620,0.620843
7,20_newsgroups(4),CLS_distilbert-base-uncased,LinearSVM,0.614295,0.614,0.614356
3,20_newsgroups(4),CLS_bert-base-uncased,LinearSVM,0.579518,0.579,0.579519
4,emotion,CLS_distilbert-base-uncased,LogisticRegression,0.409485,0.516,0.517699
5,emotion,CLS_distilbert-base-uncased,LinearSVM,0.380336,0.491,0.493848
0,emotion,CLS_bert-base-uncased,LogisticRegression,0.375942,0.459,0.463341
1,emotion,CLS_bert-base-uncased,LinearSVM,0.363656,0.442,0.451072


# Часть 2. CLS из fine-tuned модели -> внешний классификатор

In [23]:
fine_tuned_cls_results = []

# Emotion fine-tuned model
emotion_ft_model = FINE_TUNED_MODELS["emotion"]
print("Emotion fine-tuned model:", emotion_ft_model)

X_train_ft_emotion = get_cls_embeddings(
    emotion_train_texts,
    emotion_ft_model,
    max_length=128,
    batch_size=BATCH_SIZE
)

X_test_ft_emotion = get_cls_embeddings(
    emotion_test_texts,
    emotion_ft_model,
    max_length=128,
    batch_size=BATCH_SIZE
)

fine_tuned_cls_results.extend(
    train_external_classifiers(
        X_train_ft_emotion,
        emotion_train_labels,
        X_test_ft_emotion,
        emotion_test_labels,
        dataset_name="emotion",
        representation_name=f"CLS_finetuned_{emotion_ft_model}"
    )
)

print(
    "20_newsgroups fine-tuned CLS is skipped: "
    "rjac/bert-20news-classification has TensorFlow-only weights "
    "and cannot be loaded with PyTorch AutoModel in this notebook."
)

fine_tuned_cls_df = pd.DataFrame(fine_tuned_cls_results).sort_values(
    ["dataset", "f1_macro"],
    ascending=[True, False]
)

fine_tuned_cls_df

Emotion fine-tuned model: bhadresh-savani/bert-base-uncased-emotion


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


20_newsgroups fine-tuned CLS is skipped: rjac/bert-20news-classification has TensorFlow-only weights and cannot be loaded with PyTorch AutoModel in this notebook.


,dataset,representation,classifier,f1_macro,f1_micro,f1_weighted
0,emotion,CLS_finetuned_bhadresh-savani/bert-base-uncase...,LogisticRegression,0.884078,0.927,0.926665
1,emotion,CLS_finetuned_bhadresh-savani/bert-base-uncase...,LinearSVM,0.880011,0.924,0.923828


# Часть 3. Fine-tuned BERT как готовый классификатор

In [25]:
direct_ft_results = []

# Emotion direct fine-tuned classifier
emotion_preds_direct = predict_with_finetuned_model(
    emotion_test_texts,
    FINE_TUNED_MODELS["emotion"],
    label_names=emotion_label_names,
    max_length=128,
    batch_size=BATCH_SIZE
)

direct_ft_results.append({
    "dataset": "emotion",
    "representation": f"direct_finetuned_{FINE_TUNED_MODELS['emotion']}",
    "classifier": "built-in classifier",
    "f1_macro": f1_score(
        emotion_test_labels,
        emotion_preds_direct,
        average="macro",
        labels=list(range(len(emotion_label_names)))
    ),
    "f1_micro": f1_score(
        emotion_test_labels,
        emotion_preds_direct,
        average="micro",
        labels=list(range(len(emotion_label_names)))
    ),
    "f1_weighted": f1_score(
        emotion_test_labels,
        emotion_preds_direct,
        average="weighted",
        labels=list(range(len(emotion_label_names)))
    ),
})

print(
    "20_newsgroups direct fine-tuned classifier is skipped: "
    "rjac/bert-20news-classification has TensorFlow-only weights."
)

direct_ft_df = pd.DataFrame(direct_ft_results).sort_values(
    ["dataset", "f1_macro"],
    ascending=[True, False]
)

direct_ft_df

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model id2label: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


Direct classifier: bhadresh-savani/bert-base-uncased-emotion:   0%|          | 0/63 [00:00<?, ?it/s]

20_newsgroups direct fine-tuned classifier is skipped: rjac/bert-20news-classification has TensorFlow-only weights.


,dataset,representation,classifier,f1_macro,f1_micro,f1_weighted
0,emotion,direct_finetuned_bhadresh-savani/bert-base-unc...,built-in classifier,0.888943,0.929,0.928498


# Fine-tuning

In [26]:
RUN_OPTIONAL_FINE_TUNING = False

In [27]:
if RUN_OPTIONAL_FINE_TUNING:
    from torch.utils.data import Dataset, DataLoader
    from sklearn.model_selection import train_test_split

    class TransformerTextDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_length=128):
            self.texts = list(texts)
            self.labels = list(labels)
            self.tokenizer = tokenizer
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt"
            )
            item = {k: v.squeeze(0) for k, v in enc.items()}
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    ft_model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(ft_model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        ft_model_name,
        num_labels=len(emotion_label_names)
    ).to(device)

    # Берём ограниченную часть данных, чтобы fine-tuning был быстрым.
    ft_train_texts, ft_val_texts, ft_train_labels, ft_val_labels = train_test_split(
        emotion_train_texts,
        emotion_train_labels,
        test_size=0.15,
        random_state=SEED,
        stratify=emotion_train_labels,
    )

    train_ds = TransformerTextDataset(ft_train_texts, ft_train_labels, tokenizer, max_length=128)
    val_ds = TransformerTextDataset(ft_val_texts, ft_val_labels, tokenizer, max_length=128)

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    best_val_f1 = 0
    EPOCHS = 1

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Fine-tuning epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].numpy()
                batch = {k: v.to(device) for k, v in batch.items()}
                logits = model(**batch).logits
                pred = torch.argmax(logits, dim=1).cpu().numpy()
                preds.extend(pred)
                targets.extend(labels)

        val_f1 = f1_score(targets, preds, average="macro")
        print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, val_f1_macro={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model.save_pretrained("my_distilbert_emotion_finetuned")
            tokenizer.save_pretrained("my_distilbert_emotion_finetuned")

    print("Best validation F1 macro:", best_val_f1)

# Финальное сравнение всех подходов

In [28]:
final_results_df = pd.concat(
    [base_results_df, fine_tuned_cls_df, direct_ft_df],
    ignore_index=True
)

final_results_df = final_results_df.sort_values(["dataset", "f1_macro"], ascending=[True, False])
final_results_df

,dataset,representation,classifier,f1_macro,f1_micro,f1_weighted
0,20_newsgroups(4),CLS_distilbert-base-uncased,LogisticRegression,0.643699,0.643,0.643449
1,20_newsgroups(4),CLS_bert-base-uncased,LogisticRegression,0.620788,0.620,0.620843
2,20_newsgroups(4),CLS_distilbert-base-uncased,LinearSVM,0.614295,0.614,0.614356
3,20_newsgroups(4),CLS_bert-base-uncased,LinearSVM,0.579518,0.579,0.579519
10,emotion,direct_finetuned_bhadresh-savani/bert-base-unc...,built-in classifier,0.888943,0.929,0.928498
8,emotion,CLS_finetuned_bhadresh-savani/bert-base-uncase...,LogisticRegression,0.884078,0.927,0.926665
9,emotion,CLS_finetuned_bhadresh-savani/bert-base-uncase...,LinearSVM,0.880011,0.924,0.923828
4,emotion,CLS_distilbert-base-uncased,LogisticRegression,0.409485,0.516,0.517699
5,emotion,CLS_distilbert-base-uncased,LinearSVM,0.380336,0.491,0.493848
6,emotion,CLS_bert-base-uncased,LogisticRegression,0.375942,0.459,0.463341


In [29]:
print("Лучшие результаты по каждому датасету:")
best_by_dataset = final_results_df.loc[
    final_results_df.groupby("dataset")["f1_macro"].idxmax()
].sort_values("dataset")

best_by_dataset

Лучшие результаты по каждому датасету:


,dataset,representation,classifier,f1_macro,f1_micro,f1_weighted
0,20_newsgroups(4),CLS_distilbert-base-uncased,LogisticRegression,0.643699,0.643,0.643449
10,emotion,direct_finetuned_bhadresh-savani/bert-base-unc...,built-in classifier,0.888943,0.929,0.928498
